# 🇰🇭 Fine-tune Khmer LLM — QLoRA on SEA-LION 8B (you2show repos)

QLoRA fine-tuning (4-bit) លើ `you2show/Llama-SEA-LION-v3-8B-IT-bucket` (copy សាធារណៈរបស់អ្នក —
មិនត្រូវ gated login) ដោយប្រើ distillation dataset របស់អ្នក។ រត់លើ **Colab/Kaggle T4 ឥតគិតថ្លៃ**។

**ហេតុអ្វី SEA-LION?** pretrain ជាមួយខ្មែរផ្ទាល់ → base ខ្មែរខ្លាំង, ត្រូវ data តិចជាង។
**រយៈពេល**: ~30-60 នាទី / 3 epochs (subset)។ HF profile៖ https://huggingface.co/you2show

## ជំហានទី ០ — បើក GPU
- **Colab**: Runtime → Change runtime type → **T4 GPU**
- **Kaggle**: Settings ⚙️ → Accelerator → **GPU T4 x2** ឬ **P100**

In [ ]:
# ដំឡើង library (~2-3 នាទី)
!pip install -q -U "transformers>=4.46" "trl>=0.24" peft bitsandbytes accelerate datasets sentencepiece

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "| VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1e9,1),"GB")
else:
    print("⚠️  គ្មាន GPU — ត្រឡប់ទៅជំហានទី ០ បើក GPU សិន")

## Config — កែត្រង់នេះ

In [ ]:
# model bucket សាធារណៈរបស់អ្នក (មិនត្រូវ gated login)
MODEL_ID = "you2show/Llama-SEA-LION-v3-8B-IT-bucket"
# distillation dataset របស់អ្នក
DATASET_ID = "you2show/GPT-5.5-Gemini-3.1-Pro-Grok-4-Claude-Fable-5-Mythos-5-Qwen-3.7-Max-and-more-Distillation-Dataset"
OUTPUT_DIR = "./sealion-khmer-lora"
MAX_LENGTH = 1024
NUM_EPOCHS = 3
PER_DEVICE_BATCH_SIZE = 2
GRAD_ACCUMULATION = 8      # effective batch = 16
LEARNING_RATE = 2e-4
MAX_TRAIN_EXAMPLES = 500   # None = ទាំងអស់ (ចាប់ផ្តើមតូចដើម្បីតេស្ត pipeline)

## (ជម្រើស) Login Hugging Face

Repo `-bucket` សាធារណៈ → ជាទូទៅ **មិនត្រូវ login**។ ដោះ comment តែពេល repo ជា private
ឬពេលប្តូរទៅ model gated (ឧ. `aisingapore/…` ដើម)។

In [ ]:
# from huggingface_hub import login; login()

## ជំហានទី ១ — ទាញ dataset + មើលទម្រង់ពិត

In [ ]:
raw_dataset = load_dataset(DATASET_ID)
print(raw_dataset)
SPLIT = "train" if "train" in raw_dataset else list(raw_dataset.keys())[0]
print("\nប្រើ split:", SPLIT)
print("Columns:", raw_dataset[SPLIT].column_names)
print("\n--- ឧទាហរណ៍ទី ០ ---")
print(raw_dataset[SPLIT][0])

## ជំហានទី ២ — tokenizer + model (4-bit)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # T4/P100 មិនគាំទ្រ bf16 ពេញលេញ
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto",
)
model.config.use_cache = False
print("Model loaded ✅")

## ជំហានទី ៣ — Format data (auto-detect schema)

Function ខាងក្រោម **ស្គាល់ទម្រង់ច្រើនដោយស្វ័យប្រវត្តិ** — chat `messages`, ShareGPT `conversations`,
Alpaca `instruction/input/output`, `prompt/completion`, `question/answer`, ឬ `text` ស្រាប់។
វា print ទម្រង់ដែលរកឃើញ។ បើ dataset របស់អ្នកប្រើ column ខុសពីនេះ សូមបន្ថែមក្នុង `to_messages`។

In [ ]:
ROLE_MAP = {"human":"user","user":"user","gpt":"assistant","assistant":"assistant",
            "system":"system","bot":"assistant","model":"assistant"}

def to_messages(ex):
    # 1) chat messages ស្រាប់
    if ex.get("messages"):
        return ex["messages"]
    # 2) ShareGPT conversations
    if ex.get("conversations"):
        out=[]
        for t in ex["conversations"]:
            out.append({"role": ROLE_MAP.get(str(t.get("from","user")).lower(),"user"),
                        "content": t.get("value") or t.get("content") or ""})
        return out
    # 3) Alpaca / prompt-completion / Q-A
    for uk in ("instruction","prompt","question","input_text","query"):
        if ex.get(uk):
            user=str(ex[uk]).strip()
            extra=str(ex.get("input") or "").strip()
            if extra and uk!="input":
                user=f"{user}\n\n{extra}"
            for ak in ("output","completion","answer","response","output_text","chosen"):
                if ex.get(ak) is not None:
                    return [{"role":"user","content":user},
                            {"role":"assistant","content":str(ex[ak]).strip()}]
    return None

def format_example(ex):
    msgs = to_messages(ex)
    if not msgs:
        # 4) fallback៖ text ស្រាប់
        if ex.get("text"):
            return {"text": str(ex["text"])}
        return {"text": ""}
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}

data = raw_dataset[SPLIT]
if MAX_TRAIN_EXAMPLES:
    data = data.select(range(min(MAX_TRAIN_EXAMPLES, len(data))))

formatted_dataset = data.map(format_example, remove_columns=data.column_names)
formatted_dataset = formatted_dataset.filter(lambda r: len(r["text"].strip()) > 0)

print("សរុប example ត្រឹមត្រូវ:", len(formatted_dataset))
assert len(formatted_dataset) > 0, "❌ format មិនចេញ text — មើល column នៅ cell ១ រួចបន្ថែមក្នុង to_messages"
print("\n--- ឧទាហរណ៍បន្ទាប់ពី format ---\n", formatted_dataset[0]["text"][:800])

## ជំហានទី ៤ — LoRA

In [ ]:
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## ជំហានទី ៥ — Trainer

⚠️ `trl` ប្តូរ API ញឹកញាប់។ បើ `TypeError` អំពី `tokenizer=`/`dataset_text_field=`/`max_seq_length=`
នោះ version ថ្មីប្តូរឈ្មោះ។ កូដនេះប្រើ syntax បច្ចុប្បន្ន (SFTConfig + `processing_class`)។

In [ ]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_length=MAX_LENGTH,
    packing=False,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    gradient_checkpointing=True,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    fp16=True,
    optim="paged_adamw_8bit",
    logging_steps=10,
    save_strategy="epoch",
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    report_to="none",
)
trainer = SFTTrainer(
    model=model, args=training_args,
    train_dataset=formatted_dataset, processing_class=tokenizer,
)

## ជំហានទី ៦ — Train 🚀

In [ ]:
trainer.train()

## ជំហានទី ៧ — រក្សាទុក adapter

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("រក្សាទុករួច:", OUTPUT_DIR)
# រក្សាទុកអចិន្ត្រៃយ៍ទៅ HF (កុំឲ្យ Colab លុប)៖
# model.push_to_hub("you2show/sealion-khmer-lora")

## ជំហានទី ៨ — សាកល្បង

In [ ]:
model.config.use_cache = True
model.eval()
msgs=[{"role":"user","content":"សូមណែនាំរបៀបធ្វើម្ហូបខ្មែរសាមញ្ញមួយមុខ"}]
inputs=tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
with torch.no_grad():
    out=model.generate(inputs, max_new_tokens=256, do_sample=True, temperature=0.7)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

## ជំហានទី ៩ — ប្រើក្នុង A2I

Training រក្សាទុកតែ **LoRA adapter** (តូច)។ ដើម្បីប្រើក្នុង A2I ជា GGUF ត្រូវ **reload base fp16**
សិន (កុំ `merge_and_unload` លើ 4-bit ដែលទើប train), រួច merge → convert៖

```python
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
merged = PeftModel.from_pretrained(base, OUTPUT_DIR).merge_and_unload()
merged.save_pretrained("sealion-khmer-merged"); tokenizer.save_pretrained("sealion-khmer-merged")

!git clone https://github.com/ggerganov/llama.cpp && pip install -q -r llama.cpp/requirements.txt
!python llama.cpp/convert_hf_to_gguf.py sealion-khmer-merged --outfile sealion-khmer.gguf --outtype q8_0
!./llama.cpp/llama-quantize sealion-khmer.gguf sealion-khmer-q4_k_m.gguf q4_k_m
```

⚠️ 8B fp16 merge ត្រូវ ~16GB RAM — លើសពី Colab free។ Merge លើម៉ាស៊ីនធំ (A100/local),
ឬ push adapter ទៅ HF មុន។ បន្ទាប់មក ដាក់ GGUF នៅ `a2i-core/models/model.gguf` → `./run.sh`
(មើល `a2i-train/README.md`)។ ឬ serve GGUF លើ Colab តាម `a2i-core/serve_gguf_colab.ipynb`
រួចភ្ជាប់ A2I web។